# See the impact of uncorrelated structure on the measurements in the flamingo data

In [3]:
%load_ext autoreload
%autoreload 2

Loop through each filter and all the models and determine the threshold values for each

In [4]:
from get_model_probabilities import *
from add_shear_to_data import *
import scienceplots
from sidm_inference_on_data_and_models import infer_sidm
plt.style.use(["science","grid"])

Source redshift:1.36


In [30]:
imonte=10

#target_test_acc = pkl.load(open("pickles/target_test_accuracy.pkl","rb"))

ifilter='concat'
all_models =  glob(f"../models/concat/cdan_baha2dark_pre_squeezenet1_aw_1_pad_shear_avgpool_gauss_seed_*_nob1_ft_tweaked_align_10_best.pth")

#get the flamingo dz=10 results
fiducial_name = f"pickles/flamingo_tng_results.pkl"
dz10_results = pkl.load(open(fiducial_name,'rb'))

#now do the rest
output_name = f"pickles/flamingo_dz_results.pkl"

if os.path.isfile(output_name):
    all_results = pkl.load(open(output_name,'rb'))
else:
    all_results = {}


domain_lists = [ f'../data/100/obs/{ifilter}/flamingo_dz{i}.pkl' for i in [10,25,50,100, 250,500]]

for domains in domain_lists:
    
    print(domains)
    
    dzname = domains.split('/')[-1].split('_')[1].split('.')[0]
    if not dzname in all_results.keys():
        all_results[dzname] = {}
        
    for imodel in tqdm(all_models):

        seed = imodel.split('_')[-7]

        args.jwst_filter = ifilter
        args.apply_intrinsic_ell = 1.
        args.verbose=False
        args.train_split =0.05
        zs = {
        'f115w':1.6,
        'f150w':1.65,
        'concat':1.65
        }

        domain = {
        'tgt':'flamingo_obs',
        'src':'tng_obs'
        }
        args.ignore_dataset = [''] # Although i ignored during training i want to see duringn testing.

        if f"seed_{seed}" not in all_results[dzname].keys():

            all_results[dzname][f"seed_{seed}"]  = []
            

        args.unbalance = True

        for i in range(len( all_results[dzname][f"seed_{seed}"]), imonte):
            args.zs = zs[ifilter]

            target_domain = domains
            results = get_probabilities( 
                    target_domain,
                    [imodel],
                    args,
                    quiet=True
            )
            del results['data_loaders']

            all_results[dzname][f"seed_{seed}"].append( results )

        pkl.dump(all_results, open(output_name,"wb"))


../data/100/obs/concat/flamingo_dz10.pkl


100%|███████████████████████████████████████████| 30/30 [00:04<00:00,  7.22it/s]


../data/100/obs/concat/flamingo_dz25.pkl


100%|███████████████████████████████████████████| 30/30 [00:04<00:00,  7.26it/s]


../data/100/obs/concat/flamingo_dz50.pkl


...

100%|███████████████████████████████████████████| 30/30 [00:04<00:00,  7.23it/s]


../data/100/obs/concat/flamingo_dz250.pkl


100%|███████████████████████████████████████████| 30/30 [00:04<00:00,  7.17it/s]


../data/100/obs/concat/flamingo_dz500.pkl


100%|███████████████████████████████████████████| 30/30 [12:44<00:00, 25.49s/it]


In [1]:

correction = 2.
fig = plt.figure(figsize=(4,3))

ax = plt.gca()


dz_results =  f"pickles/flamingo_dz_results.pkl"


    

all_results = pkl.load(open(dz_results,'rb'))



all_means = []
all_errors = []
for idz, dz in enumerate(all_results.keys()):
    all_thresholds = []
    for imodel in all_results[dz].keys():



        tgt = get_threshold_for_cross( 
            all_results[dz][imodel], 
            function=np.mean,
            quiet=False)


        all_thresholds.append(tgt['thresholds'])

    all_thresholds = np.array(all_thresholds)
    means = np.nanmean(all_thresholds,axis=0)
    errors = np.std(all_thresholds,axis=0) / all_thresholds.shape[0]**0.38*correction
    all_means.append(1-means[0])
    all_errors.append(errors[0])
ax.errorbar( [ 2*float(i[2:]) for i in np.sort(list(all_results.keys()))], all_means/all_means[0], all_errors/all_means[0], capsize=2, fmt='o')
ax.set_xlabel("Integration length [Mpc]")
ax.set_ylabel("Model Output")

NameError: name 'plt' is not defined